## 1. Event Definitions and Empirical Probabilities

To begin our probabilistic analysis, we define two events based on the patient-level analytic dataset ($N = 1,083$):
- **Event A**: The patient has Late-Stage cancer (AJCC Stage III or IV).
- **Event B**: The patient has High MKI67 expression (defined as log1p(TPM) strictly greater than the cohort median).

**Path B requirement (ب۵) - Independence of Observations:** 
All probabilities calculated here treat the patient as the fundamental unit of observation. Because we enforced "Rule A" in Phase 1 (one primary-tumor sample per patient), no patient appears more than once in these calculations. Thus, the empirical probabilities are not artificially inflated by repeated measures.

### Marginal, Joint, and Conditional Probabilities

Using the empirical counts from the dataset:
- **Total sample size:** $N = 1,083$
- **Marginal probability of Late-Stage:** $P(A) = \frac{268}{1083} \approx 0.247$
- **Marginal probability of High MKI67:** $P(B) = \frac{541}{1083} \approx 0.500$ *(Note: Because High MKI67 is defined by the median split, it is mathematically expected to be ~50%).*
- **Joint probability (Both Late-Stage AND High MKI67):** $P(A \cap B) = \frac{131}{1083} \approx 0.121$

From these, we can calculate the conditional probability:
- **$P(B \mid A)$:** The probability of High MKI67 given the patient is Late-Stage. 
  $$P(B \mid A) = \frac{P(A \cap B)}{P(A)} = \frac{0.121}{0.247} \approx 0.489$$

### Interpretation
The probability of having High MKI67 in the general sample is $P(B) \approx 0.500$. However, when we condition on the patient having Late-Stage cancer, the probability drops slightly to $P(B \mid A) \approx 0.489$. The difference between these two values is extremely small ($\approx 0.01$), suggesting that knowing a patient's stage does not significantly change our expectation about their MKI67 expression.

In [1]:
import pandas as pd
import numpy as np

# 1. Load the data
at = pd.read_csv("../data/processed/analysis_table.csv")

# (Recreate the columns just in case)
def stage4(s):
    if pd.isna(s): return pd.NA
    for num in ["IV", "III", "II", "I"]:
        if str(s).startswith(f"Stage {num}"): return num
    return pd.NA

at["stage4"] = at["diagnoses.ajcc_pathologic_stage"].apply(stage4)
at["stage2"] = at["stage4"].map({"I": "Early", "II": "Early", "III": "Late", "IV": "Late"})
at["log_MKI67"] = np.log1p(pd.to_numeric(at["MKI67_tpm"], errors="coerce"))
at["high_MKI67"] = at["log_MKI67"] > at["log_MKI67"].median()

# 2. Calculate the counts
N = len(at)
count_A = len(at[at["stage2"] == "Late"])
count_B = len(at[at["high_MKI67"] == True])
count_A_and_B = len(at[(at["stage2"] == "Late") & (at["high_MKI67"] == True)])

# 3. Calculate the probabilities
P_A = count_A / N
P_B = count_B / N
P_A_and_B = count_A_and_B / N
P_B_given_A = count_A_and_B / count_A

print(f"Total N = {N}")
print(f"P(A) [Late Stage]       = {P_A:.3f}")
print(f"P(B) [High MKI67]       = {P_B:.3f}")
print(f"P(A and B) [Both]       = {P_A_and_B:.3f}")
print(f"P(B | A) [High given Late] = {P_B_given_A:.3f}")

# 4. Independence Check
difference = abs(P_B_given_A - P_B)
print(f"\nDifference between P(B|A) and P(B): {difference:.4f}")

Total N = 1083
P(A) [Late Stage]       = 0.247
P(B) [High MKI67]       = 0.500
P(A and B) [Both]       = 0.121
P(B | A) [High given Late] = 0.489

Difference between P(B|A) and P(B): 0.0107
